# Per-gene / per-region TWAS weights against a pre-built QtlDataset

## Description

For a single study's `QtlDataset` RDS, run `pecotmr::twasWeightsPipeline(methods = "default", ...)` per fan-out unit (gene or region). The default method preset gives the standard univariate methods (SuSiE / lasso / elastic-net / etc. as defined by `pecotmr::.twasMethodLookup("default")`).

Optionally pass a pre-fit `--fine-mapping-result` RDS; SuSiE-family TWAS methods reuse those fits via the `fineMappingResult` cache.

## Inputs

- `--qtl-dataset` &mdash; path to the QtlDataset RDS produced by `qtl_dataset.ipynb`.
- `--genes ID1 ID2 …` **or** `--regions chr:start-end …` (mutually exclusive).
- `--cis-window` &mdash; bp window (gene mode). Default 1,000,000.
- `--fine-mapping-result` &mdash; optional pre-fit FineMappingResult RDS.
- `--modular-script-dir` &mdash; directory containing the per-gene worker R scripts. Default `code/script`.

## Output

- `{cwd}/{study}.{gene|region}.twas_weights.rds` per fan-out unit.


## Example

```bash
sos run pipeline/twas_weights.ipynb twas_weights \
    --cwd output \
    --modular-script-dir /path/to/xqtl-protocol/code/script \
    --study TEST_STUDY \
    --qtl-dataset output/TEST_STUDY.qtl_dataset.rds \
    --genes ENSG00000060237 ENSG00000234593
```


In [ ]:
[global]
parameter: cwd = path('output')
parameter: study = str
parameter: qtl_dataset = path
parameter: modular_script_dir = path('code/script')
# Mutually exclusive fan-out sources: pass one as a list, leave the other empty.
parameter: genes = []
parameter: regions = []
parameter: cis_window = 1000000
parameter: fine_mapping_result = path('.')
parameter: container = ''
parameter: job_size = 1
parameter: walltime = '30m'
parameter: mem = '8G'
parameter: numThreads = 1


In [ ]:
[twas_weights]
if bool(genes) == bool(regions):
    raise ValueError("Specify exactly one of --genes (trait IDs) or --regions (chr:start-end strings).")
fanout_items = genes if genes else regions
fanout_kind  = 'gene' if genes else 'region'
def _fanout_safe(s):
    return s.replace(':', '_').replace('-', '_')
input: qtl_dataset, for_each = 'fanout_items'
output: f"{cwd}/{study}.{_fanout_safe(_fanout_items)}.twas_weights.rds"
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f"{step_name}_{_output:bn}"
bash: expand = '${ }', stderr = f"{_output}.stderr", stdout = f"{_output}.stdout", container = container
    Rscript ${modular_script_dir}/pecotmr_integration/twas_weights.R \
        --qtl-dataset ${_input} \
        ${('--gene-id ' + _fanout_items + ' --cis-window ' + str(cis_window)) if fanout_kind == 'gene' else ('--region ' + _fanout_items)} \
        --fine-mapping-result ${fine_mapping_result if fine_mapping_result.is_file() else '""'} \
        --output ${_output}
